In [2]:
import json
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
EPOCHS = 10
BATCH_SIZE = 2048
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.set_num_threads(max(1, min(4, torch.get_num_threads())))

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "data" / "creditcard.csv").exists()
        and (path / "results" / "split_indices.npz").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate project data and results/split_indices.npz. "
        "Run Notebook 01 first."
    )

RESULTS_DIR = PROJECT_ROOT / "results"
df = pd.read_csv(PROJECT_ROOT / "data" / "creditcard.csv")
with np.load(RESULTS_DIR / "split_indices.npz") as split_indices:
    train_idx = split_indices["train_idx"]
    validation_idx = split_indices["validation_idx"]

train_df = df.loc[train_idx].copy()
validation_df = df.loc[validation_idx].copy()
feature_cols = [column for column in df.columns if column != "Class"]
X_train = train_df.loc[train_df["Class"].eq(0), feature_cols]
y_val = validation_df["Class"].to_numpy()
X_val = validation_df[feature_cols]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_val_scaled = scaler.transform(X_val).astype(np.float32)

class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, input_dim),
        )

    def forward(self, values):
        return self.network(values)


def reconstruction_scores(model, values):
    model.eval()
    scores = []
    with torch.no_grad():
        for start in range(0, len(values), BATCH_SIZE):
            batch = torch.from_numpy(values[start : start + BATCH_SIZE])
            errors = (model(batch) - batch).pow(2).mean(dim=1)
            scores.append(errors.cpu().numpy())
    return np.concatenate(scores)


def evaluate_scores(y_true, scores, threshold):
    predictions = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true, predictions, labels=[0, 1]
    ).ravel()
    return {
        "threshold": float(threshold),
        "precision": float(precision_score(y_true, predictions, zero_division=0)),
        "recall": float(recall_score(y_true, predictions, zero_division=0)),
        "f1": float(f1_score(y_true, predictions, zero_division=0)),
        "average_precision": float(average_precision_score(y_true, scores)),
        "false_positive_rate": float(fp / (fp + tn)) if (fp + tn) else 0.0,
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp),
    }


model = Autoencoder(X_train_scaled.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_function = nn.MSELoss()
train_tensor = torch.from_numpy(X_train_scaled)
train_loader = DataLoader(
    TensorDataset(train_tensor),
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    samples_seen = 0
    for (batch,) in train_loader:
        optimizer.zero_grad()
        loss = loss_function(model(batch), batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch)
        samples_seen += len(batch)
    print(f"Epoch {epoch + 1}/{EPOCHS} - loss: {total_loss / samples_seen:.6f}")

val_scores = reconstruction_scores(model, X_val_scaled)
precision, recall, thresholds = precision_recall_curve(y_val, val_scores)
if len(thresholds) == 0:
    raise ValueError("No validation thresholds available.")
f1_scores = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-12)
)
best_index = int(np.argmax(f1_scores))
threshold = float(thresholds[best_index])
validation_metrics = evaluate_scores(y_val, val_scores, threshold)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
weights_path = RESULTS_DIR / "autoencoder_state.pt"
scaler_path = RESULTS_DIR / "autoencoder_scaler.joblib"
config_path = RESULTS_DIR / "autoencoder_config.json"
torch.save(model.state_dict(), weights_path)
joblib.dump(scaler, scaler_path)
with config_path.open("w", encoding="utf-8") as file:
    json.dump(
        {
            "random_seed": SEED,
            "feature_cols": feature_cols,
            "architecture": {"hidden_dims": [16, 8]},
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "threshold_selection": "Maximum validation F1",
            "validation_metrics": validation_metrics,
        },
        file,
        indent=2,
    )
print("Validation metrics:")
print(json.dumps(validation_metrics, indent=2))
print(f"Saved autoencoder weights, scaler, and config to {RESULTS_DIR}")


Epoch 1/10 - loss: 0.977379
Epoch 2/10 - loss: 0.840461
Epoch 3/10 - loss: 0.741062
Epoch 4/10 - loss: 0.700616
Epoch 5/10 - loss: 0.669337
Epoch 6/10 - loss: 0.645298
Epoch 7/10 - loss: 0.618644
Epoch 8/10 - loss: 0.586754
Epoch 9/10 - loss: 0.569984
Epoch 10/10 - loss: 0.556960
Validation metrics:
{
  "threshold": 17.924938201904297,
  "precision": 0.4,
  "recall": 0.35135135135135137,
  "f1": 0.37410071942446044,
  "average_precision": 0.2772584165636798,
  "false_positive_rate": 0.0009144840199779585,
  "true_negatives": 42608,
  "false_positives": 39,
  "false_negatives": 48,
  "true_positives": 26
}
Saved autoencoder weights, scaler, and config to c:\Users\user\Documents\AlgomaU courses\Computer Science\Third Year\Data Science with Python\Data Science Final Project\results
